In [1]:
# ============================================================
# NOTEBOOK 10 — CELL 1
# LOAD ORIGINAL TRAINING AND TEST DATA
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import joblib
import warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from imblearn.over_sampling import SMOTENC
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
print("Libraries imported successfully.")

# ============================================================
# CELL 2 — DEFINE DIRECTORIES
# ============================================================
BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed_data"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
PROCESSED_DIR = Path("../data/processed")

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Project directory:", BASE_DIR)
print("Processed data:", PROCESSED_DIR)
print("Models:", MODELS_DIR)
print("Results:", RESULTS_DIR)



Libraries imported successfully.
Project directory: c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks
Processed data: ..\data\processed
Models: c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks\models
Results: c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks\results


In [2]:
print("=" * 80)
print("NOTEBOOK 10 — FINAL SP-XGBOOST EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

print("\nProcessed data directory:")
print(PROCESSED_DIR)

# ------------------------------------------------------------
# Load ORIGINAL training data
# ------------------------------------------------------------

X_train_clean = pd.read_csv(
    PROCESSED_DIR / "X_train_clean.csv"
)

y_train_clean = pd.read_csv(
    PROCESSED_DIR / "y_train_clean.csv"
)

# Convert target to Series
if isinstance(y_train_clean, pd.DataFrame):
    y_train_clean = y_train_clean.iloc[:, 0]

# ------------------------------------------------------------
# Load ORIGINAL untouched test data
# ------------------------------------------------------------

X_test_clean = pd.read_csv(
    PROCESSED_DIR / "X_test_clean.csv"
)

y_test_clean = pd.read_csv(
    PROCESSED_DIR / "y_test_clean.csv"
)

# Convert target to Series
if isinstance(y_test_clean, pd.DataFrame):
    y_test_clean = y_test_clean.iloc[:, 0]

# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET SHAPES")
print("=" * 80)

print(
    "Original training X:",
    X_train_clean.shape
)

print(
    "Original training y:",
    y_train_clean.shape
)

print(
    "Original test X:",
    X_test_clean.shape
)

print(
    "Original test y:",
    y_test_clean.shape
)

# ------------------------------------------------------------
# Class distributions
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CLASS DISTRIBUTIONS")
print("=" * 80)

print("\nTraining:")
print(
    y_train_clean
    .value_counts()
    .sort_index()
)

print("\nTest:")
print(
    y_test_clean
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

print(
    "Training X missing values:",
    X_train_clean.isnull().sum().sum()
)

print(
    "Test X missing values:",
    X_test_clean.isnull().sum().sum()
)

print(
    "Training y missing values:",
    y_train_clean.isnull().sum()
)

print(
    "Test y missing values:",
    y_test_clean.isnull().sum()
)

# ------------------------------------------------------------
# Verify predictor consistency
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PREDICTOR CONSISTENCY CHECK")
print("=" * 80)

train_columns = list(
    X_train_clean.columns
)

test_columns = list(
    X_test_clean.columns
)

print(
    "Same predictor columns:",
    train_columns == test_columns
)

if train_columns != test_columns:

    missing_in_test = [
        col
        for col in train_columns
        if col not in test_columns
    ]

    extra_in_test = [
        col
        for col in test_columns
        if col not in train_columns
    ]

    print(
        "Missing from test:",
        missing_in_test
    )

    print(
        "Extra in test:",
        extra_in_test
    )

else:
    print(
        "✓ Training and test predictors "
        "are identical."
    )

print("\n✓ Notebook 10 Cell 1 completed.")

NOTEBOOK 10 — FINAL SP-XGBOOST EVALUATION

Processed data directory:
..\data\processed

DATASET SHAPES
Original training X: (455, 58)
Original training y: (455,)
Original test X: (114, 58)
Original test y: (114,)

CLASS DISTRIBUTIONS

Training:
RISK_LABEL
0    329
1    126
Name: count, dtype: int64

Test:
RISK_LABEL
0    83
1    31
Name: count, dtype: int64

MISSING VALUES
Training X missing values: 0
Test X missing values: 0
Training y missing values: 0
Test y missing values: 0

PREDICTOR CONSISTENCY CHECK
Same predictor columns: True
✓ Training and test predictors are identical.

✓ Notebook 10 Cell 1 completed.


In [3]:
# ============================================================
# NOTEBOOK 10 — CELL 2
# FINAL SMOTENC BALANCING OF TRAINING DATA
# ============================================================

from imblearn.over_sampling import SMOTENC

print("=" * 80)
print("NOTEBOOK 10 — FINAL SMOTENC TRAINING DATA")
print("=" * 80)

# ------------------------------------------------------------
# 1. Define categorical columns
# ------------------------------------------------------------

categorical_columns = [
    "ASSIGN_LATE",
    "Age_group",
    "Gender",
    "Year_study",
    "SES",
    "Financial_diff",
    "Self_risk_percep"
]

# ------------------------------------------------------------
# 2. Verify categorical columns exist
# ------------------------------------------------------------

missing_categorical = [
    col
    for col in categorical_columns
    if col not in X_train_clean.columns
]

if missing_categorical:
    raise KeyError(
        f"Missing categorical columns: "
        f"{missing_categorical}"
    )

print("\nCategorical columns:")
for col in categorical_columns:
    print(f" - {col}")

# ------------------------------------------------------------
# 3. Create categorical indices
# ------------------------------------------------------------

categorical_indices = [
    X_train_clean.columns.get_loc(col)
    for col in categorical_columns
]

print("\nCategorical indices:")
print(categorical_indices)

# ------------------------------------------------------------
# 4. Verify expected indices
# ------------------------------------------------------------

expected_indices = [
    6, 7, 8, 9, 10, 11, 15
]

if categorical_indices != expected_indices:
    raise ValueError(
        "Categorical indices do not match the "
        "expected training-data structure."
    )

print(
    "✓ Categorical indices verified."
)

# ------------------------------------------------------------
# 5. Run SMOTENC
# ------------------------------------------------------------

smotenc_final = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42,
    k_neighbors=5
)

X_train_final_balanced, y_train_final_balanced = (
    smotenc_final.fit_resample(
        X_train_clean,
        y_train_clean
    )
)

# ------------------------------------------------------------
# 6. Convert back to DataFrame / Series
# ------------------------------------------------------------

X_train_final_balanced = pd.DataFrame(
    X_train_final_balanced,
    columns=X_train_clean.columns
)

y_train_final_balanced = pd.Series(
    y_train_final_balanced,
    name="RISK_LABEL"
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SMOTENC RESULTS")
print("=" * 80)

print(
    "\nOriginal training shape:",
    X_train_clean.shape
)

print(
    "Balanced training shape:",
    X_train_final_balanced.shape
)

print("\nOriginal class distribution:")
print(
    y_train_clean   
    .value_counts()
    .sort_index()
)

print("\nBalanced class distribution:")
print(
    y_train_final_balanced
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 8. Check missing values
# ------------------------------------------------------------

print("\nMissing values after SMOTENC:")

print(
    X_train_final_balanced
    .isnull()
    .sum()
    .sum()
)

print(
    "Target missing values:",
    y_train_final_balanced.isnull().sum()
)

# ------------------------------------------------------------
# 9. Verify categorical values
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CATEGORICAL VALUE VALIDATION")
print("=" * 80)

for col in categorical_columns:

    original_values = set(
        X_train_clean[col]
        .dropna()
        .unique()
    )

    synthetic_values = set(
        X_train_final_balanced[col]
        .dropna()
        .unique()
    )

    invalid_values = (
        synthetic_values - original_values
    )

    print(
        f"\n{col}:"
    )

    print(
        "  Original values:",
        sorted(original_values)
    )

    print(
        "  Balanced values:",
        sorted(synthetic_values)
    )

    if invalid_values:
        print(
            "  ⚠ Invalid values:",
            sorted(invalid_values)
        )
    else:
        print(
            "  ✓ All categorical values valid"
        )

# ------------------------------------------------------------
# 10. Save final balanced training data
# ------------------------------------------------------------

X_train_final_balanced.to_csv(
    PROCESSED_DIR / "X_train_final_smotenc.csv",
    index=False
)

y_train_final_balanced.to_csv(
    PROCESSED_DIR / "y_train_final_smotenc.csv",
    index=False
)

print("\n" + "=" * 80)
print("SAVE STATUS")
print("=" * 80)

print(
    "✓ Final SMOTENC training data saved."
)

print(
    "\nTest data was NOT modified or used."
)

print("\n✓ Notebook 10 Cell 2 completed.")

NOTEBOOK 10 — FINAL SMOTENC TRAINING DATA

Categorical columns:
 - ASSIGN_LATE
 - Age_group
 - Gender
 - Year_study
 - SES
 - Financial_diff
 - Self_risk_percep

Categorical indices:
[6, 7, 8, 9, 10, 11, 15]
✓ Categorical indices verified.

FINAL SMOTENC RESULTS

Original training shape: (455, 58)
Balanced training shape: (658, 58)

Original class distribution:
RISK_LABEL
0    329
1    126
Name: count, dtype: int64

Balanced class distribution:
RISK_LABEL
0    329
1    329
Name: count, dtype: int64

Missing values after SMOTENC:
0
Target missing values: 0

CATEGORICAL VALUE VALIDATION

ASSIGN_LATE:
  Original values: [np.int64(0), np.int64(1), np.int64(2)]
  Balanced values: [np.int64(0), np.int64(1), np.int64(2)]
  ✓ All categorical values valid

Age_group:
  Original values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Balanced values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  ✓ All categorical values valid

Gender:
  Original values: 

In [4]:
# ============================================================
# NOTEBOOK 10 — CELL 3
# PRELIMINARY XGBOOST FOR SHAP FEATURE SELECTION
# ============================================================

import xgboost as xgb
import shap

print("=" * 80)
print("NOTEBOOK 10 — PRELIMINARY XGBOOST FOR SHAP")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare balanced training data
# ------------------------------------------------------------

X_shap_train = X_train_final_balanced.copy()
y_shap_train = y_train_final_balanced.copy()

print("\nTraining data:")
print("X shape:", X_shap_train.shape)
print("y shape:", y_shap_train.shape)

print("\nClass distribution:")
print(
    y_shap_train
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 2. Convert categorical columns to category dtype
# ------------------------------------------------------------

for col in categorical_columns:
    X_shap_train[col] = X_shap_train[col].astype("category")

print("\n✓ Categorical columns converted to category dtype.")

# ------------------------------------------------------------
# 3. Train preliminary XGBoost
# ------------------------------------------------------------

shap_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    enable_categorical=True
)

print("\nTraining preliminary XGBoost...")

shap_model.fit(
    X_shap_train,
    y_shap_train
)

print("✓ Preliminary XGBoost trained successfully.")

# ------------------------------------------------------------
# 4. Create SHAP TreeExplainer
# ------------------------------------------------------------

print("\nCreating SHAP TreeExplainer...")

explainer = shap.TreeExplainer(
    shap_model
)

# ------------------------------------------------------------
# 5. Calculate SHAP values
# ------------------------------------------------------------

print("Calculating SHAP values...")

shap_values = explainer.shap_values(
    X_shap_train
)

print("✓ SHAP values calculated.")

# ------------------------------------------------------------
# 6. Check SHAP dimensions
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SHAP DIMENSION CHECK")
print("=" * 80)

print(
    "XGBoost training shape:",
    X_shap_train.shape
)

print(
    "SHAP values shape:",
    np.asarray(shap_values).shape
)

print(
    "Number of features:",
    X_shap_train.shape[1]
)

# ------------------------------------------------------------
# 7. Calculate mean absolute SHAP importance
# ------------------------------------------------------------

mean_abs_shap = np.abs(
    shap_values
).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": X_shap_train.columns,
    "Mean_Absolute_SHAP": mean_abs_shap
})

# Sort descending
shap_importance = (
    shap_importance
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 8. Display top 20 features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 20 FEATURES BY MEAN ABSOLUTE SHAP")
print("=" * 80)

display(
    shap_importance.head(20).round(6)
)

print("\n✓ Notebook 10 Cell 3 completed.")

NOTEBOOK 10 — PRELIMINARY XGBOOST FOR SHAP

Training data:
X shape: (658, 58)
y shape: (658,)

Class distribution:
RISK_LABEL
0    329
1    329
Name: count, dtype: int64

✓ Categorical columns converted to category dtype.

Training preliminary XGBoost...
✓ Preliminary XGBoost trained successfully.

Creating SHAP TreeExplainer...
Calculating SHAP values...
✓ SHAP values calculated.

SHAP DIMENSION CHECK
XGBoost training shape: (658, 58)
SHAP values shape: (658, 58)
Number of features: 58

TOP 20 FEATURES BY MEAN ABSOLUTE SHAP


,Feature,Mean_Absolute_SHAP
0,GPA_S1,1.309723
1,Age_group,0.413725
2,LAB_AVG,0.352836
3,Study_hrs_day,0.259610
4,CLIN_AVG,0.236225
5,Concentration in self-study,0.226115
6,Adequate supervision,0.219285
7,Self-motivates,0.211021
8,Sleep_hrs,0.196478
9,Programme prepares for career,0.191292



✓ Notebook 10 Cell 3 completed.


In [5]:
# ============================================================
# NOTEBOOK 10 — CELL 4
# 95% CUMULATIVE SHAP FEATURE SELECTION
# ============================================================

print("=" * 80)
print("NOTEBOOK 10 — 95% CUMULATIVE SHAP FEATURE SELECTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Calculate total SHAP importance
# ------------------------------------------------------------

total_shap = (
    shap_importance["Mean_Absolute_SHAP"]
    .sum()
)

print("\nTotal SHAP importance:")
print(f"{total_shap:.6f}")

# ------------------------------------------------------------
# 2. Calculate relative importance
# ------------------------------------------------------------

shap_importance["Relative_Importance"] = (
    shap_importance["Mean_Absolute_SHAP"]
    / total_shap
)

# ------------------------------------------------------------
# 3. Calculate cumulative importance
# ------------------------------------------------------------

shap_importance["Cumulative_Importance"] = (
    shap_importance["Relative_Importance"]
    .cumsum()
)

# ------------------------------------------------------------
# 4. Find number of features required for 95%
# ------------------------------------------------------------

shap_threshold = 0.95

n_selected = (
    shap_importance[
        "Cumulative_Importance"
    ] < shap_threshold
).sum() + 1

# Prevent selecting more features than available
n_selected = min(
    n_selected,
    len(shap_importance)
)

# ------------------------------------------------------------
# 5. Extract selected features
# ------------------------------------------------------------

selected_features_final = (
    shap_importance
    .head(n_selected)["Feature"]
    .tolist()
)

# ------------------------------------------------------------
# 6. Create final selected training data
# ------------------------------------------------------------

X_train_shap_final = (
    X_train_final_balanced[
        selected_features_final
    ].copy()
)

# ------------------------------------------------------------
# 7. Display summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SHAP FEATURE SELECTION SUMMARY")
print("=" * 80)

print(
    f"\nOriginal predictors : "
    f"{X_train_final_balanced.shape[1]}"
)

print(
    f"Selected predictors : "
    f"{len(selected_features_final)}"
)

print(
    f"Features removed    : "
    f"{X_train_final_balanced.shape[1] - len(selected_features_final)}"
)

print(
    f"SHAP threshold      : "
    f"{shap_threshold:.0%}"
)

print(
    f"Cumulative SHAP at selected cutoff: "
    f"{shap_importance.iloc[n_selected - 1]['Cumulative_Importance']:.4f}"
)

# ------------------------------------------------------------
# 8. Display selected features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SELECTED FEATURES")
print("=" * 80)

for i, feature in enumerate(
    selected_features_final,
    start=1
):
    cumulative = (
        shap_importance
        .iloc[i - 1]
        ["Cumulative_Importance"]
    )

    print(
        f"{i:02d}. {feature:<40} "
        f"Cumulative SHAP: {cumulative:.4f}"
    )

# ------------------------------------------------------------
# 9. Check missing values
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA QUALITY CHECK")
print("=" * 80)

print(
    "Selected training shape:",
    X_train_shap_final.shape
)

print(
    "Missing values:",
    X_train_shap_final.isnull().sum().sum()
)

# ------------------------------------------------------------
# 10. Save SHAP results
# ------------------------------------------------------------

shap_importance.to_csv(
    RESULTS_DIR /
    "final_SHAP_feature_importance.csv",
    index=False
)

pd.DataFrame({
    "Selected_Feature":
        selected_features_final
}).to_csv(
    RESULTS_DIR /
    "final_SHAP_selected_features.csv",
    index=False
)

X_train_shap_final.to_csv(
    RESULTS_DIR /
    "X_train_final_SHAP.csv",
    index=False
)

print("\n✓ SHAP importance saved.")
print("✓ Selected feature list saved.")
print("✓ SHAP-selected training data saved.")

print("\n✓ Notebook 10 Cell 4 completed.")

NOTEBOOK 10 — 95% CUMULATIVE SHAP FEATURE SELECTION

Total SHAP importance:
7.565464

SHAP FEATURE SELECTION SUMMARY

Original predictors : 58
Selected predictors : 47
Features removed    : 11
SHAP threshold      : 95%
Cumulative SHAP at selected cutoff: 0.9554

SELECTED FEATURES
01. GPA_S1                                   Cumulative SHAP: 0.1731
02. Age_group                                Cumulative SHAP: 0.2278
03. LAB_AVG                                  Cumulative SHAP: 0.2744
04. Study_hrs_day                            Cumulative SHAP: 0.3088
05. CLIN_AVG                                 Cumulative SHAP: 0.3400
06. Concentration in self-study              Cumulative SHAP: 0.3699
07. Adequate supervision                     Cumulative SHAP: 0.3989
08. Self-motivates                           Cumulative SHAP: 0.4267
09. Sleep_hrs                                Cumulative SHAP: 0.4527
10. Programme prepares for career            Cumulative SHAP: 0.4780
11. Financial_diff           

In [6]:
# ============================================================
# NOTEBOOK 10 — CELL 5
# PREPARE SHAP-SELECTED DATA FOR OPTUNA
# ============================================================

print("=" * 80)
print("NOTEBOOK 10 — PREPARING SHAP-SELECTED DATA")
print("=" * 80)

# ------------------------------------------------------------
# 1. Copy SHAP-selected training data
# ------------------------------------------------------------

X_optuna_final = X_train_shap_final.copy()
y_optuna_final = y_train_final_balanced.copy()

# ------------------------------------------------------------
# 2. Verify dimensions
# ------------------------------------------------------------

print("\nX shape:")
print(X_optuna_final.shape)

print("\ny shape:")
print(y_optuna_final.shape)

# ------------------------------------------------------------
# 3. Verify selected features
# ------------------------------------------------------------

print("\nNumber of selected features:")
print(len(selected_features_final))

if len(selected_features_final) != 47:
    print(
        "⚠ Note: The current SHAP procedure selected "
        f"{len(selected_features_final)} features."
    )
else:
    print(
        "✓ Exactly 47 SHAP-selected features confirmed."
    )

# ------------------------------------------------------------
# 4. Identify categorical columns among selected features
# ------------------------------------------------------------

selected_categorical_columns = [
    col
    for col in categorical_columns
    if col in selected_features_final
]

print("\nCategorical features retained by SHAP:")
for col in selected_categorical_columns:
    print(f" - {col}")

print(
    "\nNumber of categorical features retained:",
    len(selected_categorical_columns)
)

# ------------------------------------------------------------
# 5. Convert categorical variables to category dtype
# ------------------------------------------------------------

for col in selected_categorical_columns:
    X_optuna_final[col] = (
        X_optuna_final[col]
        .astype("category")
    )

print(
    "\n✓ Selected categorical variables "
    "converted to category dtype."
)

# ------------------------------------------------------------
# 6. Check numerical columns
# ------------------------------------------------------------

numerical_columns_final = [
    col
    for col in X_optuna_final.columns
    if col not in selected_categorical_columns
]

print(
    "\nNumber of numerical features:",
    len(numerical_columns_final)
)

# ------------------------------------------------------------
# 7. Check data types
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA TYPE CHECK")
print("=" * 80)

dtype_summary = pd.DataFrame({
    "Feature": X_optuna_final.columns,
    "Data_Type": [
        str(dtype)
        for dtype in X_optuna_final.dtypes
    ]
})

display(dtype_summary)

# ------------------------------------------------------------
# 8. Missing-value check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING VALUE CHECK")
print("=" * 80)

missing_total = (
    X_optuna_final
    .isnull()
    .sum()
    .sum()
)

print(
    "Missing predictor values:",
    missing_total
)

print(
    "Missing target values:",
    y_optuna_final.isnull().sum()
)

# ------------------------------------------------------------
# 9. Class distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)

print(
    y_optuna_final
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 10. Save final Optuna input data
# ------------------------------------------------------------

X_optuna_final.to_csv(
    RESULTS_DIR /
    "X_final_SHAP_selected.csv",
    index=False
)

y_optuna_final.to_csv(
    RESULTS_DIR /
    "y_final_SHAP_selected.csv",
    index=False
)

print("\n✓ Final SHAP-selected data saved.")

print("\n✓ Notebook 10 Cell 5 completed.")

NOTEBOOK 10 — PREPARING SHAP-SELECTED DATA

X shape:
(658, 47)

y shape:
(658,)

Number of selected features:
47
✓ Exactly 47 SHAP-selected features confirmed.

Categorical features retained by SHAP:
 - Age_group
 - Gender
 - Financial_diff
 - Self_risk_percep

Number of categorical features retained: 4

✓ Selected categorical variables converted to category dtype.

Number of numerical features: 43

DATA TYPE CHECK


,Feature,Data_Type
0,GPA_S1,float64
1,Age_group,category
2,LAB_AVG,float64
3,Study_hrs_day,int64
4,CLIN_AVG,float64
5,Concentration in self-study,int64
6,Adequate supervision,int64
7,Self-motivates,int64
8,Sleep_hrs,int64
9,Programme prepares for career,int64



MISSING VALUE CHECK
Missing predictor values: 0
Missing target values: 0

CLASS DISTRIBUTION
RISK_LABEL
0    329
1    329
Name: count, dtype: int64

✓ Final SHAP-selected data saved.

✓ Notebook 10 Cell 5 completed.


In [7]:
# ============================================================
# NOTEBOOK 10 — CELL 6
# OPTUNA HYPERPARAMETER OPTIMISATION
# ============================================================

import optuna
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("=" * 80)
print("NOTEBOOK 10 — OPTUNA HYPERPARAMETER OPTIMISATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------

X_optuna = X_optuna_final.copy()
y_optuna = y_optuna_final.copy()

# ------------------------------------------------------------
# 2. Create internal development split
# ------------------------------------------------------------

X_opt_train, X_opt_valid, y_opt_train, y_opt_valid = (
    train_test_split(
        X_optuna,
        y_optuna,
        test_size=0.20,
        stratify=y_optuna,
        random_state=42
    )
)

print("\nOptuna training shape:")
print(X_opt_train.shape)

print("\nOptuna validation shape:")
print(X_opt_valid.shape)

print("\nOptuna training class distribution:")
print(y_opt_train.value_counts().sort_index())

print("\nOptuna validation class distribution:")
print(y_opt_valid.value_counts().sort_index())

# ------------------------------------------------------------
# 3. Make sure categorical columns have category dtype
# ------------------------------------------------------------

for col in selected_categorical_columns:

    X_opt_train[col] = (
        X_opt_train[col]
        .astype("category")
    )

    X_opt_valid[col] = (
        X_opt_valid[col]
        .astype("category")
    )

# ------------------------------------------------------------
# 4. Define Optuna objective
# ------------------------------------------------------------

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            600
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.20,
            log=True
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.60,
            1.00
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            1.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10.0,
            log=True
        ),

        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,
        "n_jobs": -1,
        "enable_categorical": True
    }

    model = xgb.XGBClassifier(
        **params
    )

    model.fit(
        X_opt_train,
        y_opt_train,
        eval_set=[
            (X_opt_valid, y_opt_valid)
        ],
        verbose=False
    )

    validation_probability = (
        model.predict_proba(
            X_opt_valid
        )[:, 1]
    )

    auc = roc_auc_score(
        y_opt_valid,
        validation_probability
    )

    return auc


# ------------------------------------------------------------
# 5. Create Optuna study
# ------------------------------------------------------------

print("\nCreating Optuna study...")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=42
    )
)

# ------------------------------------------------------------
# 6. Run optimisation
# ------------------------------------------------------------

print("\nStarting 200 Optuna trials...")
print("This may take some time.\n")

study.optimize(
    objective,
    n_trials=200,
    show_progress_bar=True
)

# ------------------------------------------------------------
# 7. Display best result
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OPTUNA RESULTS")
print("=" * 80)

print(
    "\nNumber of completed trials:",
    len(study.trials)
)

print(
    "\nBest validation ROC-AUC:"
)

print(
    f"{study.best_value:.4f}"
)

print("\nBest parameters:")

for parameter, value in study.best_params.items():
    print(
        f"{parameter}: {value}"
    )

# ------------------------------------------------------------
# 8. Save Optuna results
# ------------------------------------------------------------

optuna_results = study.trials_dataframe()

optuna_results.to_csv(
    RESULTS_DIR /
    "Notebook10_Optuna_trials.csv",
    index=False
)

# Save best parameters
pd.DataFrame(
    [study.best_params]
).to_csv(
    RESULTS_DIR /
    "Notebook10_Optuna_best_parameters.csv",
    index=False
)

print("\n✓ Optuna trial results saved.")
print("✓ Best parameters saved.")

print("\n✓ Notebook 10 Cell 6 completed.")

[I 2026-07-30 01:07:40,295] A new study created in memory with name: no-name-6c2ab7ee-c94b-407a-970e-412a42709426


NOTEBOOK 10 — OPTUNA HYPERPARAMETER OPTIMISATION

Optuna training shape:
(526, 47)

Optuna validation shape:
(132, 47)

Optuna training class distribution:
RISK_LABEL
0    263
1    263
Name: count, dtype: int64

Optuna validation class distribution:
RISK_LABEL
0    66
1    66
Name: count, dtype: int64

Creating Optuna study...

Starting 200 Optuna trials...
This may take some time.



  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-30 01:07:41,065] Trial 0 finished with value: 0.9403122130394856 and parameters: {'n_estimators': 287, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.05808361216819946, 'reg_alpha': 0.08499808989182997, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.9403122130394856.
[I 2026-07-30 01:07:41,787] Trial 1 finished with value: 0.9375573921028466 and parameters: {'n_estimators': 454, 'max_depth': 3, 'learning_rate': 0.18276027831785724, 'min_child_weight': 9, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.18340450985343382, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 0 with value: 0.9403122130394856.
[I 2026-07-30 01:07:42,395] Trial 2 finished with value: 0.9400826446280992 and parameters: {'n_estimators': 316, 'max_depth': 5, 'learning_rate': 0.06252287916406217, 'min_

In [8]:
# ============================================================
# NOTEBOOK 10 — CELL 6A
# DISPLAY BEST OPTUNA PARAMETERS
# ============================================================

print("=" * 80)
print("NOTEBOOK 10 — BEST OPTUNA PARAMETERS")
print("=" * 80)

print("\nBest internal validation ROC-AUC:")
print(f"{study.best_value:.6f}")

print("\nBest parameters:")
for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")

print("\nNumber of completed trials:")
print(len(study.trials))

print("\n✓ Optuna parameter verification completed.")

NOTEBOOK 10 — BEST OPTUNA PARAMETERS

Best internal validation ROC-AUC:
0.960055

Best parameters:
n_estimators: 577
max_depth: 3
learning_rate: 0.057157714257439644
min_child_weight: 8
subsample: 0.8100042651950841
colsample_bytree: 0.6246135443668355
gamma: 0.3142230766241593
reg_alpha: 1.2077780306552594e-05
reg_lambda: 3.265558728963812e-05

Number of completed trials:
200

✓ Optuna parameter verification completed.


In [9]:
# ============================================================
# NOTEBOOK 10 — CELL 7
# TRAIN FINAL TUNED SP-XGBOOST
# ============================================================

import xgboost as xgb
import joblib

print("=" * 80)
print("NOTEBOOK 10 — FINAL TUNED SP-XGBOOST")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare final SHAP-selected training data
# ------------------------------------------------------------

X_final_train = X_optuna_final.copy()
y_final_train = y_optuna_final.copy()

print("\nFinal training data:")
print("X shape:", X_final_train.shape)
print("y shape:", y_final_train.shape)

print("\nClass distribution:")
print(
    y_final_train
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 2. Ensure categorical variables have category dtype
# ------------------------------------------------------------

for col in selected_categorical_columns:
    X_final_train[col] = (
        X_final_train[col]
        .astype("category")
    )

print(
    "\n✓ Categorical feature types verified."
)

# ------------------------------------------------------------
# 3. Retrieve Optuna best parameters
# ------------------------------------------------------------

best_params = study.best_params.copy()

best_params.update({
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "random_state": 42,
    "n_jobs": -1,
    "enable_categorical": True
})

print("\n" + "=" * 80)
print("FINAL MODEL PARAMETERS")
print("=" * 80)

for parameter, value in best_params.items():
    print(
        f"{parameter}: {value}"
    )

# ------------------------------------------------------------
# 4. Create final SP-XGBoost model
# ------------------------------------------------------------

SP_XGBoost_final = xgb.XGBClassifier(
    **best_params
)

# ------------------------------------------------------------
# 5. Train on ALL 658 balanced observations
# ------------------------------------------------------------

print("\nTraining final SP-XGBoost...")
print(
    "Training observations:",
    len(X_final_train)
)

SP_XGBoost_final.fit(
    X_final_train,
    y_final_train,
    verbose=False
)

print(
    "✓ Final SP-XGBoost trained successfully."
)

# ------------------------------------------------------------
# 6. Verify feature count
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL STRUCTURE")
print("=" * 80)

print(
    "Training observations:",
    X_final_train.shape[0]
)

print(
    "Predictor count:",
    X_final_train.shape[1]
)

print(
    "Expected SHAP-selected predictors:",
    len(selected_features_final)
)

print(
    "Model feature count:",
    SP_XGBoost_final.n_features_in_
)

# ------------------------------------------------------------
# 7. Save final model
# ------------------------------------------------------------

model_path = (
    MODELS_DIR /
    "SP_XGBoost_final.pkl"
)

joblib.dump(
    SP_XGBoost_final,
    model_path
)

print(
    "\n✓ Final SP-XGBoost model saved:"
)

print(model_path)

print(
    "\n✓ Notebook 10 Cell 7 completed."
)

NOTEBOOK 10 — FINAL TUNED SP-XGBOOST

Final training data:
X shape: (658, 47)
y shape: (658,)

Class distribution:
RISK_LABEL
0    329
1    329
Name: count, dtype: int64



✓ Categorical feature types verified.

FINAL MODEL PARAMETERS
n_estimators: 577
max_depth: 3
learning_rate: 0.057157714257439644
min_child_weight: 8
subsample: 0.8100042651950841
colsample_bytree: 0.6246135443668355
gamma: 0.3142230766241593
reg_alpha: 1.2077780306552594e-05
reg_lambda: 3.265558728963812e-05
objective: binary:logistic
eval_metric: auc
random_state: 42
n_jobs: -1
enable_categorical: True

Training final SP-XGBoost...
Training observations: 658
✓ Final SP-XGBoost trained successfully.

FINAL MODEL STRUCTURE
Training observations: 658
Predictor count: 47
Expected SHAP-selected predictors: 47
Model feature count: 47

✓ Final SP-XGBoost model saved:
c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks\models\SP_XGBoost_final.pkl

✓ Notebook 10 Cell 7 completed.


In [12]:
# ============================================================
# NOTEBOOK 10 — CELL 8
# PREPARE ORIGINAL UNTOUCHED TEST DATA
# ============================================================

print("=" * 80)
print("NOTEBOOK 10 — PREPARING UNTOUCHED TEST DATA")
print("=" * 80)

# ------------------------------------------------------------
# 1. Preserve original test data
# ------------------------------------------------------------

X_test_final = X_test_clean.copy()
y_test_final = y_test_clean.copy()

print("\nOriginal test shape:")
print(X_test_final.shape)

print("\nOriginal test target shape:")
print(y_test_final.shape)

# ------------------------------------------------------------
# 2. Select ONLY the 47 SHAP-selected features
# ------------------------------------------------------------

missing_test_features = [
    feature
    for feature in selected_features_final
    if feature not in X_test_final.columns
]

if missing_test_features:
    raise KeyError(
        "The following SHAP-selected features "
        "are missing from the test data:\n"
        f"{missing_test_features}"
    )

X_test_final = X_test_final[
    selected_features_final
].copy()

print("\nSHAP-selected test shape:")
print(X_test_final.shape)

# ------------------------------------------------------------
# 3. Verify feature order
# ------------------------------------------------------------

if list(X_test_final.columns) != list(
    X_final_train.columns
):
    raise ValueError(
        "Training and test feature order do not match."
    )

print(
    "✓ Training and test feature order matches."
)

# ------------------------------------------------------------
# 4. Apply the same categorical dtypes
# ------------------------------------------------------------

for col in selected_categorical_columns:

    # Use the training categories as the reference
    training_categories = (
        X_final_train[col]
        .cat.categories
    )

    X_test_final[col] = pd.Categorical(
        X_test_final[col],
        categories=training_categories
    )

print(
    "✓ Test categorical dtypes aligned "
    "with training data."
)

# ------------------------------------------------------------
# 5. Check for missing values created by dtype alignment
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST DATA QUALITY CHECK")
print("=" * 80)

missing_test = (
    X_test_final
    .isnull()
    .sum()
    .sum()
)

print(
    "Missing values in selected test predictors:",
    missing_test
)

print(
    "Missing values in test target:",
    y_test_final.isnull().sum()
)

# ------------------------------------------------------------
# 6. Display test class distribution
# ------------------------------------------------------------

print("\nTest class distribution:")
print(
    y_test_final
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 7. Verify dimensions
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL TEST STRUCTURE")
print("=" * 80)

print(
    "Original untouched test:",
    X_test_clean.shape
)

print(
    "Test after SHAP feature selection:",
    X_test_final.shape
)

print(
    "Expected number of features:",
    len(selected_features_final)
)

# ------------------------------------------------------------
# 8. Verify no resampling occurred
# ------------------------------------------------------------

print("\nSMOTENC applied to test data: NO")
print("SHAP fitted on test data: NO")
print("Test data used during training: NO")
print("Test data used during Optuna: NO")

print(
    "\n✓ Original test observations remain untouched."
)

print(
    "\n✓ Notebook 10 Cell 8 completed."
)

NOTEBOOK 10 — PREPARING UNTOUCHED TEST DATA

Original test shape:
(114, 58)

Original test target shape:
(114,)

SHAP-selected test shape:
(114, 47)
✓ Training and test feature order matches.
✓ Test categorical dtypes aligned with training data.

TEST DATA QUALITY CHECK
Missing values in selected test predictors: 0
Missing values in test target: 0

Test class distribution:
RISK_LABEL
0    83
1    31
Name: count, dtype: int64

FINAL TEST STRUCTURE
Original untouched test: (114, 58)
Test after SHAP feature selection: (114, 47)
Expected number of features: 47

SMOTENC applied to test data: NO
SHAP fitted on test data: NO
Test data used during training: NO
Test data used during Optuna: NO

✓ Original test observations remain untouched.

✓ Notebook 10 Cell 8 completed.


In [13]:
# ============================================================
# NOTEBOOK 10 — CELL 9
# FINAL SP-XGBOOST PREDICTIONS ON UNTOUCHED TEST DATA
# ============================================================

print("=" * 80)
print("NOTEBOOK 10 — FINAL SP-XGBOOST TEST PREDICTIONS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Generate probability predictions
# ------------------------------------------------------------

print("\nGenerating test-set probability predictions...")

y_test_probability = (
    SP_XGBoost_final
    .predict_proba(
        X_test_final
    )[:, 1]
)

# ------------------------------------------------------------
# 2. Generate class predictions
# ------------------------------------------------------------

y_test_prediction = (
    y_test_probability >= 0.50
).astype(int)

# ------------------------------------------------------------
# 3. Verify prediction dimensions
# ------------------------------------------------------------

print("\nPrediction dimensions:")

print(
    "Actual labels:",
    y_test_final.shape
)

print(
    "Predicted probabilities:",
    y_test_probability.shape
)

print(
    "Predicted classes:",
    y_test_prediction.shape
)

# ------------------------------------------------------------
# 4. Display probability range
# ------------------------------------------------------------

print("\nProbability range:")

print(
    f"Minimum probability: "
    f"{y_test_probability.min():.6f}"
)

print(
    f"Maximum probability: "
    f"{y_test_probability.max():.6f}"
)

print(
    f"Mean probability: "
    f"{y_test_probability.mean():.6f}"
)

# ------------------------------------------------------------
# 5. Prediction class distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PREDICTED CLASS DISTRIBUTION")
print("=" * 80)

prediction_distribution = pd.Series(
    y_test_prediction,
    name="Predicted_RISK_LABEL"
).value_counts().sort_index()

print(
    prediction_distribution
)

# ------------------------------------------------------------
# 6. Actual versus predicted distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ACTUAL VS PREDICTED DISTRIBUTION")
print("=" * 80)

comparison_distribution = pd.DataFrame({
    "Actual": y_test_final.value_counts().sort_index(),
    "Predicted": pd.Series(
        y_test_prediction
    ).value_counts().sort_index()
})

comparison_distribution.index.name = (
    "RISK_LABEL"
)

print(
    comparison_distribution
)

# ------------------------------------------------------------
# 7. Create prediction dataframe
# ------------------------------------------------------------

test_predictions_df = pd.DataFrame({
    "Actual_RISK_LABEL": y_test_final.values,
    "Predicted_RISK_LABEL": y_test_prediction,
    "Predicted_Probability": y_test_probability
})

# ------------------------------------------------------------
# 8. Save predictions
# ------------------------------------------------------------

test_predictions_df.to_csv(
   RESULTS_DIR /
    "SP_XGBoost_final_test_predictions.csv",
    index=False
)

print(
    "\n✓ Test predictions saved."
)

# ------------------------------------------------------------
# 9. Final integrity statement
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST-SET INTEGRITY")
print("=" * 80)

print(
    "Test observations evaluated:",
    len(X_test_final)
)

print(
    "SMOTENC applied to test:",
    "NO"
)

print(
    "Test observations used during training:",
    "NO"
)

print(
    "Test observations used during Optuna:",
    "NO"
)

print(
    "Test observations used for SHAP fitting:",
    "NO"
)

print(
    "\n✓ Final predictions generated on untouched test data."
)

print(
    "\n✓ Notebook 10 Cell 9 completed."
)

NOTEBOOK 10 — FINAL SP-XGBOOST TEST PREDICTIONS

Generating test-set probability predictions...

Prediction dimensions:
Actual labels: (114,)
Predicted probabilities: (114,)
Predicted classes: (114,)

Probability range:
Minimum probability: 0.001371
Maximum probability: 0.985094
Mean probability: 0.293452

PREDICTED CLASS DISTRIBUTION
Predicted_RISK_LABEL
0    86
1    28
Name: count, dtype: int64

ACTUAL VS PREDICTED DISTRIBUTION
            Actual  Predicted
RISK_LABEL                   
0               83         86
1               31         28

✓ Test predictions saved.

TEST-SET INTEGRITY
Test observations evaluated: 114
SMOTENC applied to test: NO
Test observations used during training: NO
Test observations used during Optuna: NO
Test observations used for SHAP fitting: NO

✓ Final predictions generated on untouched test data.

✓ Notebook 10 Cell 9 completed.


In [14]:
# ============================================================
# NOTEBOOK 10 — CELL 10
# FINAL SP-XGBOOST TEST PERFORMANCE
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("=" * 80)
print("NOTEBOOK 10 — FINAL SP-XGBOOST TEST PERFORMANCE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Calculate performance metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test_final,
    y_test_prediction
)

precision = precision_score(
    y_test_final,
    y_test_prediction,
    zero_division=0
)

recall = recall_score(
    y_test_final,
    y_test_prediction,
    zero_division=0
)

f1 = f1_score(
    y_test_final,
    y_test_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test_final,
    y_test_probability
)

# ------------------------------------------------------------
# 2. Display final metrics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL TEST PERFORMANCE")
print("=" * 80)

print(
    f"\nAccuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1-score : {f1:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc:.4f}"
)

# ------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test_final,
    y_test_prediction
)

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(
    "\n                 Predicted"
)

print(
    "                 0       1"
)

print(
    f"Actual 0     {cm[0,0]:5d}   {cm[0,1]:5d}"
)

print(
    f"Actual 1     {cm[1,0]:5d}   {cm[1,1]:5d}"
)

# ------------------------------------------------------------
# 4. Classification report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test_final,
        y_test_prediction,
        target_names=[
            "Not At-Risk",
            "At-Risk"
        ],
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 5. Save final metrics
# ------------------------------------------------------------

final_metrics_df = pd.DataFrame({
    "Model": [
        "Final SP-XGBoost"
    ],
    "Accuracy": [
        accuracy
    ],
    "Precision": [
        precision
    ],
    "Recall": [
        recall
    ],
    "F1": [
        f1
    ],
    "ROC_AUC": [
        roc_auc
    ]
})

final_metrics_df.to_csv(
    RESULTS_DIR /
    "SP_XGBoost_final_test_metrics.csv",
    index=False
)

# ------------------------------------------------------------
# 6. Save confusion matrix
# ------------------------------------------------------------

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual_0_Not_At_Risk",
        "Actual_1_At_Risk"
    ],
    columns=[
        "Predicted_0_Not_At_Risk",
        "Predicted_1_At_Risk"
    ]
)

cm_df.to_csv(
    RESULTS_DIR /
    "SP_XGBoost_final_confusion_matrix.csv"
)

# ------------------------------------------------------------
# 7. Final integrity statement
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL EVALUATION STATUS")
print("=" * 80)

print(
    "Test observations:",
    len(y_test_final)
)

print(
    "Test set resampled:",
    "NO"
)

print(
    "Test set used during feature selection:",
    "NO"
)

print(
    "Test set used during Optuna:",
    "NO"
)

print(
    "Test set used during model training:",
    "NO"
)

print(
    "\n✓ Final SP-XGBoost evaluation completed."
)

print(
    "\n✓ Metrics saved successfully."
)

NOTEBOOK 10 — FINAL SP-XGBOOST TEST PERFORMANCE

FINAL TEST PERFORMANCE

Accuracy : 0.6404
Precision: 0.3214
Recall   : 0.2903
F1-score : 0.3051
ROC-AUC  : 0.5799

CONFUSION MATRIX

                 Predicted
                 0       1
Actual 0        64      19
Actual 1        22       9

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Not At-Risk     0.7442    0.7711    0.7574        83
     At-Risk     0.3214    0.2903    0.3051        31

    accuracy                         0.6404       114
   macro avg     0.5328    0.5307    0.5312       114
weighted avg     0.6292    0.6404    0.6344       114


FINAL EVALUATION STATUS
Test observations: 114
Test set resampled: NO
Test set used during feature selection: NO
Test set used during Optuna: NO
Test set used during model training: NO

✓ Final SP-XGBoost evaluation completed.

✓ Metrics saved successfully.
